<a href="https://colab.research.google.com/github/Dilandds/CNN-emotion-detection/blob/main/CCN_emotion_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# check gpu
import torch

print(torch.__version__)
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# kaggle no longer gives a downloadable json, paste creds instead
import json, os
from getpass import getpass

username = input("kaggle username: ")
key = getpass("kaggle key: ")

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": username, "key": key}, f)

!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d msambare/fer2013
!unzip -q fer2013.zip -d /content/fer2013

In [ ]:
# sanity check - counts per class
import os

DATA_DIR = "/content/fer2013"

for split in ["train", "test"]:
    split_dir = os.path.join(DATA_DIR, split)
    classes = sorted(os.listdir(split_dir))
    print(f"\n{split} ({len(classes)} classes)")
    total = 0
    for c in classes:
        n = len(os.listdir(os.path.join(split_dir, c)))
        total += n
        print(f"  {c:10s} {n}")
    print(f"  total: {total}")

In [ ]:
# look at some actual images
import matplotlib.pyplot as plt
from PIL import Image
import random

classes = sorted(os.listdir(os.path.join(DATA_DIR, "train")))

fig, axes = plt.subplots(2, 7, figsize=(14, 4))
for i, c in enumerate(classes):
    folder = os.path.join(DATA_DIR, "train", c)
    files = os.listdir(folder)
    for row in range(2):
        img = Image.open(os.path.join(folder, random.choice(files)))
        axes[row, i].imshow(img, cmap="gray")
        axes[row, i].axis("off")
        if row == 0:
            axes[row, i].set_title(c)
plt.tight_layout()
plt.show()

In [ ]:
# check size / mode of a random image
sample_path = os.path.join(DATA_DIR, "train", "happy", os.listdir(os.path.join(DATA_DIR, "train", "happy"))[0])
img = Image.open(sample_path)
print("size:", img.size)
print("mode:", img.mode)

In [ ]:
# class balance
counts = {c: len(os.listdir(os.path.join(DATA_DIR, "train", c))) for c in classes}

plt.figure(figsize=(8, 4))
plt.bar(counts.keys(), counts.values())
plt.title("train class balance")
plt.ylabel("num images")
plt.show()

counts